![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

## Processing of recordings containing HFOV during neonatal transport among cases: `AT000001-AT002190`

#### Author: Dr Gusztav Belteki
##### email: gbelteki@aol.com

### 1. Import the required libraries and set options

In [1]:
import IPython
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as dates

import os
import sys
import pickle

from pandas import Series, DataFrame
from datetime import datetime, timedelta

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('mode.chained_assignment', None) 

In [2]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 08:03:38) [Clang 14.0.6 ]
pandas version: 2.2.2
matplotlib version: 3.9.2
NumPy version: 1.26.4
IPython version: 9.11.0


### 2. List and set the working directory and the directory to write out data

In [3]:
# Topic of the Notebook which will also be the name of the subfolder containing results
TOPIC = 'HFOV_2026'

# Path to project folder containing clinical data (current weights only) and for export of results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Name of the external hard drive
DRIVE = 'GB_1'

# Processed data loaded from external drive
DIR_READ = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian_new')

# Results will be written in this folder
DIR_WRITE =  os.path.join(os.sep, PATH, 'ventilation_fabian_new', 'Analyses', TOPIC)
os.makedirs(DIR_WRITE, exist_ok=True)

# Images and raw data will be written on an external hard drive
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian_new', TOPIC)
os.makedirs(DATA_DUMP, exist_ok=True)

In [4]:
DIR_READ, DIR_WRITE, DATA_DUMP

('/Volumes/GB_1/data_dump/fabian_new',
 '/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian_new/Analyses/HFOV_2026',
 '/Volumes/GB_1/data_dump/fabian_new/HFOV_2026')

### 3. Import ventilator data from pickle archives

In [5]:
# Import ventilator parameters, settings and alarms (AT000001 - AT002100)
with open(os.path.join(DIR_READ, 'data_pars_measurements_trimmed_new.pickle'), 'rb') as handle:
    data_pars_measurements_1 = pickle.load(handle)
with open(os.path.join(DIR_READ, 'data_pars_settings_trimmed_new.pickle'), 'rb') as handle:
    data_pars_settings_1 = pickle.load(handle)
with open(os.path.join(DIR_READ, 'data_pars_alarms_trimmed_new.pickle'), 'rb') as handle:
    data_pars_alarms_1 = pickle.load(handle)

# Import ventilator parameters, settings and alarms (AT002101 - )
with open(os.path.join(DIR_READ, 'data_pars_measurements_trimmed_new_2.pickle'), 'rb') as handle:
    data_pars_measurements_2 = pickle.load(handle)
with open(os.path.join(DIR_READ, 'data_pars_settings_trimmed_new_2.pickle'), 'rb') as handle:
    data_pars_settings_2 = pickle.load(handle)
with open(os.path.join(DIR_READ, 'data_pars_alarms_trimmed_new_2.pickle'), 'rb') as handle:
    data_pars_alarms_2 = pickle.load(handle)

In [6]:
data_pars_measurements = {**data_pars_measurements_1, **data_pars_measurements_2}
data_pars_settings = {**data_pars_settings_1, **data_pars_settings_2}
data_pars_alarms = {**data_pars_alarms_1, **data_pars_alarms_2}
len(data_pars_measurements), len(data_pars_settings), len(data_pars_alarms)

(1743, 1743, 1743)

In [7]:
sorted(data_pars_measurements.keys())[-5:]

['AT002183', 'AT002185', 'AT002186', 'AT002187', 'AT002190']

In [8]:
# Import DataFrame with ventilation modes

# AT000001 - AT002100
with open(os.path.join(DIR_READ, 'vent_modes_trimmed_new.pickle'), 'rb') as handle:
    vent_modes_1 = pickle.load(handle)
# AT002101 - 
with open(os.path.join(DIR_READ, 'vent_modes_trimmed_new_2.pickle'), 'rb') as handle:
    vent_modes_2 = pickle.load(handle)

vent_modes = pd.concat([vent_modes_1, vent_modes_2], sort = False)
len(vent_modes)

1743

In [9]:
# Import clinical data
with open(os.path.join(DIR_READ, 'clin_df_new.pickle'), 'rb') as handle:
    clin_df = pickle.load(handle)
len(clin_df)

1774

### 4. Shift the time stamp of ventilator recordings recorded with incorrect time stamps

Az eltérés valóban az AT000110-estől kezdődik és az AT000216-ig tart. Sajnos volt közben egy téli/nyári váltás is. 
Március 28 után a plusz 9 órával kell korrigálni, előtte 10 órával. 

In [10]:
data_pars_measurements['AT000110'].head()

,PIP,MAP,PEEP,Cdyn,C20_C,R,MV,VTemand,Leak,RR,Trigger,VTimand,FiO2,TC,RR_CO2,MV_kg,VTimand_kg,VTemand_kg,Cdyn_kg
2021-01-26 23:37:01,40.9,5.2,4.9,30.1,87.7,205.6,0.374,11.0,0.0,35.0,2.0,11.3,30.8,6.19,255,0.287692,8.692308,8.461538,23.153846
2021-01-26 23:37:03,29.1,6.5,5.0,26.3,76.7,232.8,0.418,11.6,0.0,35.0,2.0,12.2,30.7,6.14,255,0.321538,9.384615,8.923077,20.230769
2021-01-26 23:37:05,17.7,8.5,4.9,20.2,59.8,252.4,0.227,9.7,0.0,35.0,2.0,10.8,30.7,5.12,255,0.174615,8.307692,7.461538,15.538462
2021-01-26 23:37:07,17.4,9.2,4.9,17.8,52.6,260.1,0.190,8.9,0.0,35.0,2.0,9.7,30.7,4.63,255,0.146154,7.461538,6.846154,13.692308
2021-01-26 23:37:09,18.3,9.2,4.9,15.6,46.4,267.3,0.159,8.2,0.0,35.0,2.0,8.9,30.8,4.18,255,0.122308,6.846154,6.307692,12.000000


In [11]:
for case in data_pars_measurements:
    # print(int(case[2:].lstrip('0')))
    if 110 <= int(case[2:].lstrip('0')) < 195:
        data_pars_measurements[case].index = data_pars_measurements[case].index.shift(10, freq='h')
        data_pars_settings[case].index = data_pars_settings[case].index.shift(10, freq='h')
        data_pars_alarms[case].index = data_pars_alarms[case].index.shift(10, freq='h')
        
    elif 195 <= int(case[2:].lstrip('0')) <= 216:
        data_pars_measurements[case].index = data_pars_measurements[case].index.shift(9, freq='h')
        data_pars_settings[case].index = data_pars_settings[case].index.shift(9, freq='h')
        data_pars_alarms[case].index = data_pars_alarms[case].index.shift(9, freq='h')

In [12]:
data_pars_measurements['AT000110'].head()

,PIP,MAP,PEEP,Cdyn,C20_C,R,MV,VTemand,Leak,RR,Trigger,VTimand,FiO2,TC,RR_CO2,MV_kg,VTimand_kg,VTemand_kg,Cdyn_kg
2021-01-27 09:37:01,40.9,5.2,4.9,30.1,87.7,205.6,0.374,11.0,0.0,35.0,2.0,11.3,30.8,6.19,255,0.287692,8.692308,8.461538,23.153846
2021-01-27 09:37:03,29.1,6.5,5.0,26.3,76.7,232.8,0.418,11.6,0.0,35.0,2.0,12.2,30.7,6.14,255,0.321538,9.384615,8.923077,20.230769
2021-01-27 09:37:05,17.7,8.5,4.9,20.2,59.8,252.4,0.227,9.7,0.0,35.0,2.0,10.8,30.7,5.12,255,0.174615,8.307692,7.461538,15.538462
2021-01-27 09:37:07,17.4,9.2,4.9,17.8,52.6,260.1,0.190,8.9,0.0,35.0,2.0,9.7,30.7,4.63,255,0.146154,7.461538,6.846154,13.692308
2021-01-27 09:37:09,18.3,9.2,4.9,15.6,46.4,267.3,0.159,8.2,0.0,35.0,2.0,8.9,30.8,4.18,255,0.122308,6.846154,6.307692,12.000000


### 5. Identify which recordings had HFOV mode

In [13]:
vent_modes_hfov = vent_modes[vent_modes['HFO'] > 0]
len(vent_modes_hfov)

148

In [14]:
vent_modes_hfov.sort_values(['HFO'])

Ventilator_mode,CPAP,DUOPAP,HFO,IPPV,NCPAP,O2therapy,PSV,SIMV,SIMVPSV,SIPPV,ventilation,noninvasive,total,VG
AT000394,0.0,0.0,2.0,0.0,0.0,3146.0,0.0,0.0,0.0,0.0,2.0,3146.0,3148,0
AT001056,0.0,0.0,8.0,0.0,0.0,4790.0,0.0,0.0,0.0,0.0,8.0,4790.0,4798,0
AT001824,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,8272.0,8280.0,0.0,8280,5200
AT000717,0.0,0.0,10.0,0.0,0.0,1558.0,0.0,0.0,0.0,0.0,10.0,1558.0,1568,0
AT000953,0.0,0.0,10.0,0.0,0.0,1658.0,0.0,0.0,0.0,0.0,10.0,1658.0,1668,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AT001865,0.0,0.0,10548.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10548.0,0.0,10548,2288
AT001854,0.0,0.0,10614.0,0.0,0.0,0.0,0.0,0.0,0.0,128.0,10742.0,0.0,10742,122
AT000952,0.0,0.0,11376.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11376.0,0.0,11376,0
AT001464,16.0,0.0,12224.0,0.0,0.0,0.0,0.0,2356.0,0.0,0.0,14580.0,16.0,14596,12358


In [15]:
# Removes those recordings which have <1 minute HFOV
vent_modes_hfov = vent_modes_hfov[vent_modes_hfov['HFO'] > 60]
len(vent_modes_hfov)

138

In [16]:
# How many had conventional ventilation as well
conv_modes = ['IPPV', 'PSV', 'SIMV', 'SIMVPSV', 'SIPPV']
vent_modes_hfov['conventional'] = vent_modes_hfov[conv_modes].sum(axis=1)

In [17]:
vent_modes_hfov.head()

Ventilator_mode,CPAP,DUOPAP,HFO,IPPV,NCPAP,O2therapy,PSV,SIMV,SIMVPSV,SIPPV,ventilation,noninvasive,total,VG,conventional
AT000033,0.0,0.0,2430.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2430.0,0.0,2430,2424,0.0
AT000034,0.0,0.0,1654.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1654.0,0.0,1654,1654,0.0
AT000048,0.0,0.0,2844.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2844.0,0.0,2844,0,0.0
AT000081,0.0,0.0,872.0,0.0,0.0,0.0,0.0,0.0,0.0,2548.0,3420.0,0.0,3420,3368,2548.0
AT000086,0.0,0.0,2042.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2042.0,0.0,2042,0,0.0


### 6. Limit data to cases who were ventilated using HFOV for at least part of the transfer

In [18]:
# Limit ventilator recordings to hfov
data_pars_measurements_hfov = {key:value for key, value in data_pars_measurements.items() if key in vent_modes_hfov.index}
data_pars_settings_hfov = {key:value for key, value in data_pars_settings.items() if key in vent_modes_hfov.index}
data_pars_alarms_hfov = {key:value for key, value in data_pars_alarms.items() if key in vent_modes_hfov.index}

In [19]:
len(data_pars_measurements_hfov), len(data_pars_settings_hfov), len(data_pars_alarms_hfov)

(138, 138, 138)

### 7. Clean up ventilator data

#### A. Clean up ventilator parameters

In [20]:
for case in data_pars_measurements_hfov:
    # Remove completely empty columns of parameters which are not relevant
    data_pars_measurements_hfov[case].dropna(how = 'all', axis = 1, inplace = True)
    
    # Make sure the type of all remaining data is float
    data_pars_measurements_hfov[case] = data_pars_measurements_hfov[case].astype('float')

In [21]:
data_pars_measurements_hfov['AT000033'].head(2)

,PIP,MAP,MV,Leak,VTimand,deltaP,VThf_emand,DCO2,HFOV_freq,FiO2,RR_CO2,MV_kg,VTimand_kg,VThf_emand_kg,DCO2_kg2
2020-11-20 09:24:31,34.5,16.0,1.795,34.0,3.9,33.9,3.8,144.0,10.0,97.1,255.0,1.196667,2.600000,2.533333,64.0
2020-11-20 09:24:33,33.2,15.1,1.823,34.0,3.8,32.4,3.8,144.0,10.0,97.1,255.0,1.215333,2.533333,2.533333,64.0


#### B. Clean up ventilator settings

In [22]:
data_pars_settings_hfov['AT000033'].head()

,Patient_range,Ventilator_mode,PIP_set,PEEP_set,FiO2_set,Flow_insp_set,Flow_exp_set,deltaP_set,Freq_set_HFOV,MAP_set_HFOV,Volume_limit_set,VG_set,Powerstate,Battery_rem_time,Battery_rem_pc,MV_lim_high_set,MV_lim_low_set,PIP_lim_high_set,PEEP_lim_low_set,Leakage_lim_set,DCO2_lim_high_set,DCO2_lim_low_set,etCO2_lim_high_set,etCO2_lim_low_set,Measuring_unit_pressure_set,Measuring_unit_CO2_set,CO2_baropressure_set,Flow_sensor_state,Oxy_sensor_state,P_man_breath_CPAP_HFO_set,P_man_breath_duoPAP_NCPAP_set,FiO2_flush_time_set,FiO2_flush_set,Ventilation_stopped,VG_state,Volume_limit_state,Ventilator_range,Trigger_mode,I_E_HFOV,Pressure_rise_control,HFO_recruitment,HFO_flow,Bias_flow,Trigger_mode_2,FOT_running,Leak_comp,VG_set_kg,MV_lim_high_set_kg,MV_lim_low_set_kg,DCO2_lim_high_set_kg2,DCO2_lim_low_set_kg2
2020-11-20 09:24:31,Neonatal,HFO,15.0,NaN,100,10.0,NaN,35.0,10.0,15.0,off,3.6,Battery,134.0,97,2.9,0.15,62.2,NaN,off,1500,1,60.0,8.0,cmH2O,mmHg,759,on,on,15.0,NaN,60,100,no,on,off,0,Volumetrigger,0.33,I-flow,off,on,off,Flowtrigger,no,off,2.4,1.933333,0.1,666.666667,0.444444
2020-11-20 09:24:33,Neonatal,HFO,15.0,NaN,100,10.0,NaN,35.0,10.0,15.0,off,3.6,Battery,134.0,97,2.9,0.15,62.2,NaN,off,1500,1,60.0,8.0,cmH2O,mmHg,759,on,on,15.0,NaN,60,100,no,on,off,0,Volumetrigger,0.33,I-flow,off,on,off,Flowtrigger,no,off,2.4,1.933333,0.1,666.666667,0.444444
2020-11-20 09:24:35,Neonatal,HFO,15.0,NaN,100,10.0,NaN,35.0,10.0,15.0,off,3.6,Battery,134.0,97,2.9,0.15,62.2,NaN,off,1500,1,60.0,8.0,cmH2O,mmHg,759,on,on,15.0,NaN,60,100,no,on,off,0,Volumetrigger,0.33,I-flow,off,on,off,Flowtrigger,no,off,2.4,1.933333,0.1,666.666667,0.444444
2020-11-20 09:24:37,Neonatal,HFO,15.0,NaN,100,10.0,NaN,35.0,10.0,15.0,off,3.6,Battery,137.0,97,2.9,0.15,62.2,NaN,off,1500,1,60.0,8.0,cmH2O,mmHg,759,on,on,15.0,NaN,60,100,no,on,off,0,Volumetrigger,0.33,I-flow,off,on,off,Flowtrigger,no,off,2.4,1.933333,0.1,666.666667,0.444444
2020-11-20 09:24:39,Neonatal,HFO,15.0,NaN,100,10.0,NaN,35.0,10.0,15.0,off,3.6,Battery,137.0,97,2.9,0.15,62.2,NaN,off,1500,1,60.0,8.0,cmH2O,mmHg,759,on,on,15.0,NaN,60,100,no,on,off,0,Volumetrigger,0.33,I-flow,off,on,off,Flowtrigger,no,off,2.4,1.933333,0.1,666.666667,0.444444


In [23]:
for case in data_pars_settings_hfov:
    # Remove completely empty columns of parameters which are not relevant
    data_pars_settings_hfov[case].dropna(how = 'all', axis = 1, inplace = True)

In [24]:
# Make categorical variables "category" type

categorical = {'Patient_range', 'Ventilator_mode', 'Powerstate', 'Measuring_unit_pressure_set', 
               'Measuring_unit_CO2_set', 'Flow_sensor_state', 'Oxy_sensor_state',
               'Ventilation_stopped', 'VG_state',  'Trigger_mode', 'I_E_HFOV', 
               'Pressure_rise_control', 'HFO_flow', 'Trigger_mode_2', 'FOT_running', 'Leak_comp'}

for case in data_pars_settings_hfov:
    to_change_to_categorical = categorical & set(data_pars_settings_hfov[case].columns)
    for par in to_change_to_categorical:
        data_pars_settings_hfov[case][par] = data_pars_settings_hfov[case][par].astype('category')

#### C. Cleanup ventilator alarms

In [25]:
for case in data_pars_alarms_hfov:
    # Remove empty columns of parameters which are not noninvasive ventilation
    data_pars_alarms_hfov[case].dropna(how = 'all', axis = 1, inplace = True)
    # Make sure the type of all remaining data is float
    data_pars_alarms_hfov[case] = data_pars_alarms_hfov[case].astype('float')

### 8. Generate DataFrames with data limited to the hfov periods

In [26]:
data_pars_settings_hfov['AT000456']['Ventilator_mode'].value_counts()

Ventilator_mode
SIPPV    2968
HFO      2131
Name: count, dtype: int64

In [27]:
### Generate DataFrames with the HFO parts only 
data_pars_settings_hfov_only = {}
data_pars_measurements_hfov_only = {}
data_pars_alarms_hfov_only = {}

for recording in data_pars_settings_hfov:
    data_pars_settings_hfov_only[recording] = \
        data_pars_settings_hfov[recording][data_pars_settings_hfov[recording]['Ventilator_mode'] == 'HFO']
    
    to_reindex_with = data_pars_settings_hfov_only[recording].index
    
    data_pars_measurements_hfov_only[recording] = data_pars_measurements_hfov[recording].reindex(to_reindex_with)
    data_pars_alarms_hfov_only[recording] = data_pars_alarms_hfov[recording].reindex(to_reindex_with)

### 9. Calculate difference between set and actual parameters

In [28]:
hfov_pars_settings = [('deltaP', 'deltaP_set'), ('HFOV_freq', 'Freq_set_HFOV'), ('FiO2', 'FiO2_set'), ('MAP', 'MAP_set_HFOV')]

for par, setting in hfov_pars_settings:
    for case in data_pars_measurements_hfov_only:
        data_pars_measurements_hfov_only[case][f'{par}_diff'] = \
            data_pars_measurements_hfov_only[case][par] - data_pars_settings_hfov_only[case][setting]

for case in data_pars_measurements_hfov_only:
    
    if 'VG_set' in data_pars_settings_hfov_only[case]:
        data_pars_measurements_hfov_only[case]['VThf_diff'] = data_pars_measurements_hfov_only[case][par] - \
            data_pars_settings_hfov_only[case]['VG_set'][data_pars_settings_hfov_only[case]['VG_set'] != 'off']
    
    if  'VG_set_kg' in data_pars_settings_hfov_only[case]: 
        data_pars_measurements_hfov_only[case]['VThf_diff_kg'] = data_pars_measurements_hfov_only[case][par] - \
            data_pars_settings_hfov_only[case]['VG_set_kg'][data_pars_settings_hfov_only[case]['VG_set_kg'] != 'off']

### 10. Export processed data as pickle arcives

In [29]:
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_hfov.pickle'), 'wb') as handle:
    pickle.dump(data_pars_measurements_hfov, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(os.path.join(DATA_DUMP, 'data_pars_settings_hfov.pickle'), 'wb') as handle:
    pickle.dump(data_pars_settings_hfov, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_hfov.pickle'), 'wb') as handle:
    pickle.dump(data_pars_alarms_hfov, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_hfov_only.pickle'), 'wb') as handle:
    pickle.dump(data_pars_measurements_hfov_only, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(os.path.join(DATA_DUMP, 'data_pars_settings_hfov_only.pickle'), 'wb') as handle:
    pickle.dump(data_pars_settings_hfov_only, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_hfov_only.pickle'), 'wb') as handle:
    pickle.dump(data_pars_alarms_hfov_only, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [30]:
with open(os.path.join(DATA_DUMP, 'vent_modes_hfov.pickle'), 'wb') as handle:
    pickle.dump(vent_modes_hfov, handle, protocol=pickle.HIGHEST_PROTOCOL)